# Fairness Evaluation
## CVD Fairness Dissertation — NB3

Purpose: Evaluate whether the model performs equitably across male and female patients.

Covers sex-stratified performance metrics, formal fairness metrics, calibration
by sex, and feature distribution comparisons between prediction groups.

Does NOT contain: SHAP explanations, waterfall plots, dependence plots.
Those are in NB1 and NB2.

## 1. Imports

In [1]:
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    accuracy_score, recall_score,
    precision_score, f1_score
)
from sklearn.calibration import calibration_curve

## 2. Paths and Constants

In [2]:
MODEL_PATH    = Path("../models/baseline_models/cardio_xgb_baseline_model.pkl")
TEST_DATA     = Path("../data/test_train_val_sets/cardio_baseline_test.csv")
CONFIG_PATH   = Path("../config/dataset_split_sizes.json")
OUTPUT_FAIR   = Path("../outputs/fairness")
OUTPUT_FAIR.mkdir(parents=True, exist_ok=True)

THRESHOLD  = 0.5
GENDER_COL = "gender"
FEMALE_VAL = 0
MALE_VAL   = 1
DROP_COLS  = ["cardio", "stratify"]

## 3. Sanity Checks

In [3]:
with open(CONFIG_PATH) as f:
    expected_sizes = json.load(f)

test_df = pd.read_csv(TEST_DATA)
X_test  = test_df.drop(columns=DROP_COLS)
y_test  = test_df["cardio"]

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

y_pred  = model.predict(X_test)
y_prob  = model.predict_proba(X_test)[:, 1]

# Check 1: test set size
assert len(test_df) == expected_sizes["test"], (
    f"Test size mismatch: got {len(test_df)}, expected {expected_sizes['test']}"
)

# Check 2: no target leakage
assert "cardio" not in X_test.columns, "cardio found in X_test"

# Check 3: no nulls
assert X_test.isnull().sum().sum() == 0, "Nulls found in X_test"

# Check 4: class balance
cvd_frac = test_df["cardio"].mean()
assert 0.48 < cvd_frac < 0.52, f"Unexpected class balance: {cvd_frac:.3f}"

# Check 5: sex distribution
female_frac = (test_df[GENDER_COL] == FEMALE_VAL).mean()
assert 0.62 < female_frac < 0.68, f"Unexpected female fraction: {female_frac:.3f}"

# Check 6: predictions computed at correct threshold
assert set(np.unique(y_pred)) <= {0, 1}, "y_pred contains values other than 0 and 1"

# Check 7: protected attribute not the only predictor
# confirms model was not trained on gender alone by checking feature count
assert X_test.shape[1] > 1, "X_test has only one feature"

print("All sanity checks passed")
print(f"  Test size   : {len(test_df):,}")
print(f"  Female      : {female_frac:.3f}")
print(f"  CVD rate    : {cvd_frac:.3f}")
print(f"  Threshold   : {THRESHOLD}")

All sanity checks passed
  Test size   : 10,227
  Female      : 0.650
  CVD rate    : 0.495
  Threshold   : 0.5


## 4. Sex Masks and Subgroup Splits

In [4]:
female_mask = test_df[GENDER_COL].values == FEMALE_VAL
male_mask   = test_df[GENDER_COL].values == MALE_VAL

X_test_female = X_test[female_mask].copy()
X_test_male   = X_test[male_mask].copy()

y_test_female = y_test.values[female_mask]
y_test_male   = y_test.values[male_mask]

y_pred_female = y_pred[female_mask]
y_pred_male   = y_pred[male_mask]

y_prob_female = y_prob[female_mask]
y_prob_male   = y_prob[male_mask]

# Masks are mutually exclusive and exhaustive
assert (female_mask & male_mask).sum() == 0, "Sex masks overlap"
assert (female_mask | male_mask).sum() == len(test_df), "Sex masks do not cover full test set"

print(f"Female patients : {female_mask.sum():,}")
print(f"Male patients   : {male_mask.sum():,}")

Female patients : 6,649
Male patients   : 3,578


## 5. Helper: get_rates

Computes TPR, TNR, FPR, FNR, and PPV from true and predicted labels.
Used for both the overall performance table and the fairness metrics.

In [5]:
def get_rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "TPR": tp / (tp + fn),
        "TNR": tn / (tn + fp),
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp),
        "PPV": tp / (tp + fp),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn
    }

## 6. Formal Fairness Metrics

EOD and disparate impact are the two primary fairness metrics used in this study,
as defined in the methodology chapter.

EOD measures the difference in true positive rate between men and women.
A positive value means the model identifies a higher proportion of male CVD
cases correctly than female.

Disparate impact measures whether women are flagged as high risk at the same
rate as men across the full population. A value below 1.0 means women are
flagged less often.

All additional metrics are computed for completeness and reported in the appendix.

Note: all fairness conclusions are specific to the 0.5 threshold used throughout.

In [6]:
rates_f = get_rates(y_test_female, y_pred_female)
rates_m = get_rates(y_test_male,   y_pred_male)

pos_rate_f = y_pred_female.mean()
pos_rate_m = y_pred_male.mean()

eod = rates_m["TPR"] - rates_f["TPR"]
di  = pos_rate_f / pos_rate_m

print("Primary fairness metrics")
print(f"  EOD (0 = fair, + = model favours men) : {eod:+.4f}")
print(f"  DI  (1 = fair, < 1 = women flagged less) : {di:.4f}")

print("\nSupporting metrics")
print(f"  Positive rate female          : {pos_rate_f:.4f}")
print(f"  Positive rate male            : {pos_rate_m:.4f}")
print(f"  Equalised odds TPR diff (M-F) : {rates_m['TPR'] - rates_f['TPR']:+.4f}")
print(f"  Equalised odds FPR diff (M-F) : {rates_m['FPR'] - rates_f['FPR']:+.4f}")
print(f"  Predictive parity PPV diff (M-F) : {rates_m['PPV'] - rates_f['PPV']:+.4f}")
print(f"  FNR parity diff (F-M)         : {rates_f['FNR'] - rates_m['FNR']:+.4f}")

fairness_table = pd.DataFrame([{
    "EOD":                    eod,
    "DI":                     di,
    "TPR_female":             rates_f["TPR"],
    "TPR_male":               rates_m["TPR"],
    "FNR_female":             rates_f["FNR"],
    "FNR_male":               rates_m["FNR"],
    "pos_rate_female":        pos_rate_f,
    "pos_rate_male":          pos_rate_m,
    "equalised_odds_TPR_diff": rates_m["TPR"] - rates_f["TPR"],
    "equalised_odds_FPR_diff": rates_m["FPR"] - rates_f["FPR"],
    "predictive_parity_diff": rates_m["PPV"] - rates_f["PPV"],
    "FNR_parity_diff":        rates_f["FNR"] - rates_m["FNR"],
}])

fairness_table.to_csv(OUTPUT_FAIR / "fairness_metrics_summary.csv", index=False)
print("\nSaved: fairness_metrics_summary.csv")

Primary fairness metrics
  EOD (0 = fair, + = model favours men) : -0.0226
  DI  (1 = fair, < 1 = women flagged less) : 1.0180

Supporting metrics
  Positive rate female          : 0.4550
  Positive rate male            : 0.4469
  Equalised odds TPR diff (M-F) : -0.0226
  Equalised odds FPR diff (M-F) : +0.0007
  Predictive parity PPV diff (M-F) : -0.0022
  FNR parity diff (F-M)         : -0.0226

Saved: fairness_metrics_summary.csv
